# Lab 2: Learning-Based Privacy Policy Analysis

![Machine Learning Model](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSt9FiuVYRGX2ejeBHqGMHgsqysI3s4bZewIw&s)

*Figure: Machine learning models used to analyze and classify privacy policy text*

## Background: What is PolicyLint (PPLint)?

So far, we have focused on analyzing individual sentences from privacy policies.

However, privacy policy analysis can also be done at the **document level**.

### What is PolicyLint?

**PolicyLint** is a research tool that analyzes an entire privacy policy and detects potential **contradictions**.

For example:
- One part of a policy may say "we do not collect location data"
- Another part may say "we collect location data for analytics"

PolicyLint identifies such inconsistencies by analyzing:
- negation (e.g., "do not collect")
- relationships between sentences

---

### Why is this important?

Privacy policies are often long and complex, and contradictions can make them:
- confusing
- misleading

---

### How does this relate to our lab?

In this lab, we focus on a simpler task:

**Sentence-level classification**  
We detect whether a sentence describes a data practice using:
- TF-IDF features  
- Machine learning models  

PolicyLint is mentioned here as a more advanced approach that analyzes **entire documents**, which you can explore in future work.

## Step 1: Setup

In this step, we install and import the required libraries for data processing, machine learning, and evaluation.

In [ ]:
!pip -q install pandas scikit-learn openpyxl

import pandas as pd
import numpy as np
import requests, io, re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

## Step 2: Load the Labeled Privacy Policy Dataset

In this step, we download and load the dataset containing app metadata and privacy policy links.

In [ ]:
xlsx_url = "https://raw.githubusercontent.com/CUSecLab/2020-ACSAC-VPA-Privacy-Policy-Analysis/main/dataset/4_actions_with_policy_1967_with_duplicate.xlsx"

response = requests.get(xlsx_url)
apps_df = pd.read_excel(io.BytesIO(response.content))

print(apps_df.shape)
print(apps_df.columns.tolist())
apps_df.head()


(1967, 6)
['catagory', 'action-link-href', 'name', 'company', 'rate', 'privacy-policy-link-href']


,catagory,action-link-href,name,company,rate,privacy-policy-link-href
0,Kids & family,https://assistant.google.com/services/a/uid/00...,Jungle Adventure,Creativity Incorporated,4.4,http://creativitymobile.com/privacypolicy/
1,Kids & family,https://assistant.google.com/services/a/uid/00...,Strangest Day Ever,Creativity Incorporated,4.1,http://creativitymobile.com/privacypolicy/
2,Business & finance,https://assistant.google.com/services/a/uid/00...,Auto Loan Advisor,"i9 Systems, Inc",5.0,http://i9systems.com/node/8
3,Business & finance,https://assistant.google.com/services/a/uid/00...,Business Insurance Advisor,"i9 Systems, Inc",5.0,http://i9systems.com/node/8
4,Business & finance,https://assistant.google.com/services/a/uid/00...,Credit Repair Advisor,"i9 Systems, Inc",5.0,http://i9systems.com/node/8



## Step 3: Keep Only Rows With a Privacy Policy Link

In this step, we remove apps that do not have a valid privacy policy link and remove duplicate links.

This ensures we only work with usable data.

In [ ]:
apps_df = apps_df.dropna(subset=["privacy-policy-link-href"]).copy()
apps_df = apps_df.drop_duplicates(subset=["privacy-policy-link-href"]).reset_index(drop=True)

print("Apps with policy links:", len(apps_df))
apps_df[["name", "company", "privacy-policy-link-href"]].head()



Apps with policy links: 1791


,name,company,privacy-policy-link-href
0,Jungle Adventure,Creativity Incorporated,http://creativitymobile.com/privacypolicy/
1,Auto Loan Advisor,"i9 Systems, Inc",http://i9systems.com/node/8
2,Tayo Ambulance,KIGLE,http://kiglestudio.com/privacy_policy/eng
3,The Bakersfield Californian,NaN,http://spokenlayer.com/privacy
4,Angry Fortune Teller,Starbutter AI,http://static.starbutter.com/StarbutterPrivacy...


## Step 4A: Filter Hard-to-Access Policy URLs

Some privacy policy links (e.g., Google Docs, Google Drive) cannot be easily scraped.

We filter out these “hard” URLs to avoid errors during data collection.

💡 Why do we sample only 200 apps?

Downloading hundreds of policies can take a long time.  
We use a smaller sample to make the lab faster and more practical.

In [ ]:
import re

def is_hard_url(url: str) -> bool:
    url = str(url).lower()
    hard_patterns = [
        "docs.google.com",     # google docs often requires access or blocks
        "drive.google.com",
        "sites.google.com",    # google sites sometimes blocks
    ]
    return any(p in url for p in hard_patterns)

apps_sample = apps_df.sample(200, random_state=42).reset_index(drop=True)
apps_sample["policy_url"] = apps_sample["privacy-policy-link-href"].astype(str)

easy_sample = apps_sample[~apps_sample["policy_url"].apply(is_hard_url)].reset_index(drop=True)

print("Original sample:", len(apps_sample))
print("Easy-to-fetch sample:", len(easy_sample))
easy_sample[["name", "policy_url"]].head()


Original sample: 200
Easy-to-fetch sample: 101


,name,policy_url
0,CVS Pharmacy,https://www.cvs.com/help/privacy_policy.jsp
1,MicroBot,https://microbot.is/privacy-policy/
2,Anime Helper,https://anime-helper.flycricket.io/privacy.html
3,Homey,https://legal.athom.com/?document=privacy-poli...
4,SG Travel Buddy,https://sg-travel-buddy-bc2b0.firebaseapp.com/...


## Step 4B: Download Privacy Policy Text

In this step, we download the actual privacy policy text from each URL.

We use web scraping techniques to:
- fetch the webpage
- remove unnecessary HTML content
- extract clean text

⚠️ Note: Some policies cannot be downloaded due to restrictions or formatting issues.

In [ ]:
import time, re
import requests
from bs4 import BeautifulSoup

def fetch_policy_text(url, timeout=12, max_chars=200000):
    url = str(url).strip()
    headers = {"User-Agent": "Mozilla/5.0"}

    # skip obvious PDFs
    if url.lower().endswith(".pdf"):
        return None

    for _ in range(2):
        try:
            r = requests.get(url, timeout=timeout, headers=headers, allow_redirects=True)
            if r.status_code != 200:
                return None

            ctype = r.headers.get("Content-Type", "").lower()
            if "pdf" in ctype:
                return None

            soup = BeautifulSoup(r.text, "html.parser")
            for tag in soup(["script", "style", "noscript"]):
                tag.decompose()

            text = soup.get_text(separator=" ")
            text = re.sub(r"\s+", " ", text).strip()

            if len(text) < 200:
                return None
            return text[:max_chars]

        except Exception:
            time.sleep(1)

    return None

easy_sample["policy_text"] = easy_sample["policy_url"].apply(fetch_policy_text)
easy_ok = easy_sample.dropna(subset=["policy_text"]).reset_index(drop=True)

print("Downloaded policies:", len(easy_ok), "out of", len(easy_sample))
easy_ok[["name", "policy_url"]].head()


Downloaded policies: 37 out of 101


,name,policy_url
0,CVS Pharmacy,https://www.cvs.com/help/privacy_policy.jsp
1,Anime Helper,https://anime-helper.flycricket.io/privacy.html
2,SG Travel Buddy,https://sg-travel-buddy-bc2b0.firebaseapp.com/...
3,Cleveland Clinic,https://my.clevelandclinic.org/about/website/p...
4,Neato Robot,https://www.neatorobotics.com/privacy-policy/


### Note on Data Collection

Not all privacy policy URLs can be downloaded automatically due to:

- Access restrictions (e.g., Google Docs, login required)
- PDF-only policies
- Anti-scraping protections

We proceed with the subset of policies that were successfully downloaded.

This reflects real-world challenges in data collection.

In [ ]:
easy_ok[["name", "policy_url"]].sample(5, random_state=1)


,name,policy_url
2,SG Travel Buddy,https://sg-travel-buddy-bc2b0.firebaseapp.com/...
29,SmartContinente,https://shelf.ai/tos
3,Cleveland Clinic,https://my.clevelandclinic.org/about/website/p...
22,Trend Micro,https://www.trendmicro.com/en_ae/about/legal/p...
25,Bitcoin Update Philippines,https://www.edmundcinco.com/projects/bitcoinup...


### Important Note on Labels (Weak Supervision)

Because labels are generated automatically using keyword rules (from Lab 1), this is called **weak supervision**.

This means:
- Labels are not manually verified
- Model performance reflects how well it learns keyword patterns
- It does NOT represent true human-annotated accuracy

### Status

So far, we have:
- Downloaded real privacy policies  
- Extracted clean text  
- Prepared data for sentence-level analysis  

Next, we will convert policies into sentences and generate labels.

## Step 5: Build Sentence-Level Dataset

In this step, we split each privacy policy into individual sentences.

This process is called **sentence tokenization** in NLP.

Each sentence will become one data point for machine learning.

In [ ]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import nltk
nltk.download("punkt")
from nltk.tokenize import sent_tokenize

rows = []
for _, r in easy_ok.iterrows():
    for s in sent_tokenize(r["policy_text"]):
        s = s.strip()
        if len(s) < 20:
            continue
        rows.append({"sentence_text": s})

sent_df = pd.DataFrame(rows)
print("Total sentences:", len(sent_df))


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Total sentences: 4370


## Step 6: Clean Sentences and Create Weak Labels

In this step, we:

1. Clean text (lowercase, remove punctuation)
2. Generate labels using keyword rules from Lab 1

This creates a dataset for training machine learning models.s

In [ ]:
import re

data_type_keywords = [
    "email", "email address", "location", "geolocation",
    "contact", "contacts", "name", "phone", "phone number",
    "payment", "credit card", "ip address", "device id", "account",
    "personal information", "personal data"
]

action_keywords = [
    "collect", "gather", "obtain", "receive",
    "use", "process", "store", "retain",
    "share", "disclose", "sell", "transfer"
]

recipient_keywords = [
    "third party", "third-party", "partners", "affiliates",
    "advertisers", "analytics providers", "service providers",
    "business partners"
]

purpose_keywords = [
    "advertising", "ads", "marketing",
    "analytics", "improve", "personalize", "recommend",
    "research", "statistics", "performance", "provide our services"
]

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def is_data_practice_sentence(text):
    has_action = any(w in text for w in action_keywords)
    has_other = any(w in text for w in (data_type_keywords + recipient_keywords + purpose_keywords))
    return int(has_action and has_other)

sent_df["clean_text"] = sent_df["sentence_text"].apply(clean_text)
sent_df["is_data_practice"] = sent_df["clean_text"].apply(is_data_practice_sentence)

print(sent_df["is_data_practice"].value_counts())
sent_df.head()


is_data_practice
0    2965
1    1405
Name: count, dtype: int64


,sentence_text,clean_text,is_data_practice
0,Privacy Policy Skip to main content Home Help ...,privacy policy skip to main content home help ...,1
1,We refer to these collectively as the “Service...,we refer to these collectively as the services...,1
2,If you are looking for information about how a...,if you are looking for information about how a...,0
3,If you use the CVS App Personal Health Record ...,if you use the cvs app personal health record ...,0
4,"By using our Services, you agree to the collec...",by using our services you agree to the collect...,0


### Note on Data Distribution

The dataset is **moderately imbalanced**, with fewer data practice sentences than non–data practice sentences.

This reflects real-world privacy policies, where most sentences do not describe data collection or usage.

## Step 7: Train/Test Split and TF-IDF Feature Extraction

In this step, we prepare data for machine learning:

1. Split data into training and testing sets  
2. Convert text into numerical features using **TF-IDF**

### What is TF-IDF?

TF-IDF (Term Frequency–Inverse Document Frequency) converts text into numbers.

It measures:
- how important a word is in a sentence
- relative to all other sentences

This allows machine learning models to process text data.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = sent_df["clean_text"]
y = sent_df["is_data_practice"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print("TF-IDF shapes:", X_train_vec.shape, X_test_vec.shape)


TF-IDF shapes: (3496, 13864) (874, 13864)


## Step 8: Train a Learning-Based Model (Logistic Regression)

Logistic Regression is a simple and effective classification algorithm.

It learns patterns from labeled data and predicts whether a sentence is:
- data practice (1)
- not data practice (0)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

model = LogisticRegression(max_iter=2000)
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

print("Classification Report (Logistic Regression):\n")
print(classification_report(y_test, y_pred, digits=3))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


Classification Report (Logistic Regression):

              precision    recall  f1-score   support

           0      0.884     0.934     0.908       593
           1      0.842     0.740     0.788       281

    accuracy                          0.872       874
   macro avg      0.863     0.837     0.848       874
weighted avg      0.870     0.872     0.870       874

Confusion Matrix:

[[554  39]
 [ 73 208]]


### Results Interpretation

- High accuracy (~87%) shows strong performance  
- High precision → predictions are reliable  
- Lower recall → some data practice sentences are missed  

💡 This model is **conservative** (fewer false positives, more false negatives)

⚠️ Since labels are weak (keyword-based), results reflect learning of rules—not true ground truth.

## Step 9: Train Another Model — Naive Bayes

Naive Bayes is a simple probabilistic model.

It assumes:
👉 words are independent of each other

This makes it fast, but sometimes less accurate.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

y_pred_nb = nb_model.predict(X_test_vec)

print("Classification Report (Naive Bayes):\n")
print(classification_report(y_test, y_pred_nb, digits=3))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_nb))


Classification Report (Naive Bayes):

              precision    recall  f1-score   support

           0      0.788     0.963     0.866       593
           1      0.852     0.452     0.591       281

    accuracy                          0.799       874
   macro avg      0.820     0.707     0.729       874
weighted avg      0.808     0.799     0.778       874

Confusion Matrix:

[[571  22]
 [154 127]]


### Observations

- Good performance on non-data sentences  
- Poor recall for data practice sentences  

❗ Misses many true cases → weak for this task  

👉 Logistic Regression performs better

Naive Bayes is faster and simpler but struggles to capture complex language patterns, making it less suitable than Logistic Regression for privacy policy analysis.


## Step 10: Train Another Model — Linear SVM

Linear SVM is a powerful model for text classification.

It works well with:
- high-dimensional data  
- sparse features like TF-IDF

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

svm_model = LinearSVC()
svm_model.fit(X_train_vec, y_train)

y_pred_svm = svm_model.predict(X_test_vec)

print("Classification Report (Linear SVM):\n")
print(classification_report(y_test, y_pred_svm, digits=3))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_svm))


Classification Report (Linear SVM):

              precision    recall  f1-score   support

           0      0.924     0.921     0.922       593
           1      0.834     0.840     0.837       281

    accuracy                          0.895       874
   macro avg      0.879     0.880     0.880       874
weighted avg      0.895     0.895     0.895       874

Confusion Matrix:

[[546  47]
 [ 45 236]]


### Observations

- Highest accuracy (~89.7%)  
- Best recall for data practice sentences  
- Fewer missed cases (false negatives)  

👉 Best overall model in this lab

# Step 11: Comparative Evaluation of Learning-Based Models

In this step, we compare the performance of all learning-based models used in this lab:
- Logistic Regression
- Multinomial Naive Bayes
- Linear SVM

We focus on **F1-score for the data practice class**, as it balances precision and recall
and is most relevant for privacy policy analysis.


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
import pandas as pd

comparison_df = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "Linear SVM"
    ],
    "Precision (Data Practice)": [
        precision_score(y_test, y_pred, pos_label=1),
        precision_score(y_test, y_pred_nb, pos_label=1),
        precision_score(y_test, y_pred_svm, pos_label=1),
    ],
    "Recall (Data Practice)": [
        recall_score(y_test, y_pred, pos_label=1),
        recall_score(y_test, y_pred_nb, pos_label=1),
        recall_score(y_test, y_pred_svm, pos_label=1),
    ],
    "F1-score (Data Practice)": [
        f1_score(y_test, y_pred, pos_label=1),
        f1_score(y_test, y_pred_nb, pos_label=1),
        f1_score(y_test, y_pred_svm, pos_label=1),
    ]
})

comparison_df


,Model,Precision (Data Practice),Recall (Data Practice),F1-score (Data Practice)
0,Logistic Regression,0.842105,0.740214,0.787879
1,Naive Bayes,0.852349,0.451957,0.590698
2,Linear SVM,0.833922,0.839858,0.836879


# Step 12: Save Dataset

In [ ]:
# Save dataset for Lab 3
sent_df.to_csv("sentences_dataset.csv", index=False)

print("Dataset saved as sentences_dataset.csv")

Dataset saved as sentences_dataset.csv


## Conclusion

In this lab, we explored learning-based approaches for detecting data practice sentences.

We:
- collected real privacy policies  
- created sentence-level data  
- generated weak labels  
- trained multiple ML models  

### Key Findings

- ML models outperform simple keyword rules  
- Linear SVM performs best  
- Weak labels limit true accuracy  

### Future Work

- Use human-labeled data  
- Apply deep learning / LLMs  
- Explore document-level analysis (PolicyLint)